In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle


# Load the dataset
df = pd.read_csv("socal2.csv")
df.drop(columns=['image_id'], inplace=True)

# Merge city and street into location
df['location'] = df['citi'] + ' ' + df['street']
df.drop_duplicates(subset="location", keep="last", inplace=True)

# Encode location with LabelEncoder
le = LabelEncoder()
df['location_encoded'] = le.fit_transform(df['location'])

# Save encoding mapping
location_mapping = dict(zip(df['location_encoded'], df['location']))
with open("location_mapping.pkl", "wb") as f:
    pickle.dump(location_mapping, f)

# Normalize sqft and pricing
scaler_sqft = MinMaxScaler()
df['norm_sqft'] = scaler_sqft.fit_transform(df[['sqft']])

scaler_price = MinMaxScaler()
df['norm_price'] = scaler_price.fit_transform(df[['price']])


# Prepare training and testing data
X = df[['location_encoded', 'bed', 'bath', 'norm_sqft']]
y = df['norm_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Multiple Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

#Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

#Lasso Regression
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train, y_train)
y_pred_lasso = lasso_model.predict(X_test)

# Random Forest Model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=700,       # More trees for better learning
    learning_rate=0.05,     # Lower learning rate for stable training
    max_depth=7,            # Prevents excessive depth (overfitting)
    subsample=0.8,          # Uses 80% of data per tree (reduces variance)
    colsample_bytree=0.8,   # Uses 80% of features per tree (improves generalization)
    reg_lambda=1,           # L2 regularization (reduces overfitting)
    reg_alpha=0.1,          # L1 regularization (adds sparsity)
    gamma=0.1,              # Minimum loss reduction required for a split
    random_state=42
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)


# SModel Evaluation Function
def evaluate_model(y_test, y_pred, model_name):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    print(f"\nModel: {model_name}")
    print(f"Mean Absolute Error      : {mae:.2f}") 
    print(f"Mean Square Error        : {mse:.2f}")
    print(f"Root Mean Absolute Error : {rmse:.2f}")
    print(f"R-squared Score          : {r2:.4f}")
    return r2

# Step 8: Compare All Models
models = {
    "Linear Regression": y_pred_linear,
    "Ridge Regression": y_pred_ridge,
    "Lasso Regression": y_pred_lasso,
    "Randomforest Regression": y_pred_rf,
    "XGBoost": y_pred_xgb
}

best_model = None
best_r2 = -1

for model_name, y_pred in models.items():
    r2 = evaluate_model(y_test, y_pred, model_name)
    if r2 > best_r2:
        best_r2 = r2
        best_model = model_name

#print(f"\nBest Model: {best_model} with R² Score: {best_r2:.4f}")

# Step 9: Plot Model Predictions
plt.figure(figsize=(10, 5))
plt.scatter(y_test, y_pred_linear, label="Linear Regression", alpha=0.6)
#plt.scatter(y_test, y_pred_ridge, label="Ridge", alpha=0.6)
#plt.scatter(y_test, y_pred_lasso, label="Lasso", alpha=0.6)
#plt.scatter(y_test, y_pred_rf, label="RanForest", alpha=0.6)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.legend()
plt.title("Linear Regression Models")
plt.show()

# Step 9: Plot Model Predictions
plt.figure(figsize=(10, 5))
#plt.scatter(y_test, y_pred_linear, label="Linear Regression", alpha=0.6)
plt.scatter(y_test, y_pred_ridge, label="Ridge", alpha=0.6)
#plt.scatter(y_test, y_pred_lasso, label="Lasso", alpha=0.6)
#plt.scatter(y_test, y_pred_rf, label="RanForest", alpha=0.6)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.legend()
plt.title("Ridge ression Models")
plt.show()

# Step 9: Plot Model Predictions
plt.figure(figsize=(10, 5))
#plt.scatter(y_test, y_pred_linear, label="Linear Regression", alpha=0.6)
#plt.scatter(y_test, y_pred_ridge, label="Ridge", alpha=0.6)
plt.scatter(y_test, y_pred_lasso, label="Lasso", alpha=0.6)
#plt.scatter(y_test, y_pred_rf, label="RanForest", alpha=0.6)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.legend()
plt.title("Laso Regression Models")
plt.show()

# Step 9: Plot Model Predictions
plt.figure(figsize=(10, 5))
#plt.scatter(y_test, y_pred_linear, label="Linear Regression", alpha=0.6)
#plt.scatter(y_test, y_pred_ridge, label="Ridge", alpha=0.6)
#plt.scatter(y_test, y_pred_lasso, label="Lasso", alpha=0.6)
#plt.scatter(y_test, y_pred_rf, label="RanForest", alpha=0.6)
plt.scatter(y_test, y_pred_xgb, alpha=0.6, color="blue", label="XGBoost")

# Add diagnoal line tot help visualize perfect prediction
x = np.linspace(min(y_test), max(y_test), 100)
plt.plot(x, x, color="red", linestyle="dashed", label="Perfect Predictions")

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.legend()
plt.title("Laso Regression Models")
plt.show()